In [ ]:
"""
Environment Setup Module.
Install all required libraries. Run this cell first.
"""
!pip install qiskit[visualization] qiskit-aer qiskit-ibm-runtime matplotlib scipy pylatexenc

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter, ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer.primitives import Estimator, Sampler
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from IPython.display import display

print("\n ENVIRONMENT SETUP COMPLETE — proceed to Exercise 1.")

---
## Exercise 1: The Parameter-Shift Rule

The parameter-shift rule gives us **exact** quantum gradients from only two circuit evaluations:

$$\frac{\partial f}{\partial\theta_k} = \frac{1}{2}\left[f\!\left(\theta_k + \frac{\pi}{2}\right) - f\!\left(\theta_k - \frac{\pi}{2}\right)\right]$$

This holds whenever $f(\theta)$ is generated by a gate of the form $G(\theta) = e^{-i\theta P/2}$ with $P$ a Pauli operator.

**What you will do:**
1. Build the simplest possible case: $f(\theta) = \langle Z\rangle_{R_y(\theta)|0\rangle} = \cos\theta$.
2. Compute the gradient analytically: $df/d\theta = -\sin\theta$.
3. Compute the gradient via the parameter-shift rule.
4. Verify they match **exactly** (up to shot noise).
5. **Experiment:** observe that finite-difference approximations ($\varepsilon$-shift) are *not* exact.

In [ ]:
"""
Parameter-Shift Rule Module.
Demonstrates exact gradient computation for a single-qubit Ry circuit.
"""

def build_ry_circuit(theta_val: float) -> QuantumCircuit:
    """1-qubit Ry(theta) circuit measuring <Z>."""
    qc = QuantumCircuit(1)
    qc.ry(theta_val, 0)
    return qc

def evaluate_expectation(theta_val: float, estimator) -> float:
    """Compute <Z> for Ry(theta)|0> using the Estimator primitive."""
    qc = build_ry_circuit(theta_val)
    obs = SparsePauliOp('Z')
    job = estimator.run([(qc, obs)])
    return float(job.result()[0].data.evs)

def psr_gradient(theta_val: float, estimator) -> float:
    """
    Parameter-shift rule gradient at theta.
    grad = [f(theta + pi/2) - f(theta - pi/2)] / 2
    """
    f_plus  = evaluate_expectation(theta_val + math.pi/2, estimator)
    f_minus = evaluate_expectation(theta_val - math.pi/2, estimator)
    return (f_plus - f_minus) / 2

def finite_diff_gradient(theta_val: float, estimator, eps: float = 1e-2) -> float:
    """
    Finite-difference gradient approximation.
    grad ≈ [f(theta + eps) - f(theta - eps)] / (2*eps)
    This is an approximation (O(eps^2) error), unlike the exact PSR.
    """
    f_plus  = evaluate_expectation(theta_val + eps, estimator)
    f_minus = evaluate_expectation(theta_val - eps, estimator)
    return (f_plus - f_minus) / (2*eps)


# ── Compare PSR vs analytic vs finite-difference ────────────────────────────
est = Estimator()
thetas = np.linspace(0, 2*math.pi, 40)

f_exact    = np.cos(thetas)
df_exact   = -np.sin(thetas)
df_psr     = np.array([psr_gradient(t, est) for t in thetas])
df_fd      = np.array([finite_diff_gradient(t, est, eps=1e-2) for t in thetas])

print("Parameter-Shift Rule Verification")
print("-" * 58)
print(f"{'θ (rad)':>10}  {'Analytic':>10}  {'PSR':>10}  {'Fin.Diff':>10}  {'|PSR-Exact|':>12}")
print("-" * 58)
for i in [5, 10, 15, 20, 25, 30]:
    err = abs(df_psr[i] - df_exact[i])
    print(f"  {thetas[i]:8.4f}  {df_exact[i]:10.6f}  {df_psr[i]:10.6f}  "
          f"{df_fd[i]:10.6f}  {err:.2e}")

# ── Plot ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(thetas, f_exact, 'b-', lw=2, label='$f(\\theta) = \\cos\\theta$')
axes[0].set_title('Expectation Value $\\langle Z\\rangle_{R_y(\\theta)}$', fontsize=12)
axes[0].set_xlabel('$\\theta$ (rad)'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(thetas, df_exact, 'g-', lw=2.5, label='Analytic: $-\\sin\\theta$')
axes[1].plot(thetas, df_psr,   'r--', lw=2,  label='Parameter-Shift Rule (exact)')
axes[1].plot(thetas, df_fd,    'k:', lw=1.5, label='Finite Difference ($\\varepsilon=0.01$)')
axes[1].set_title('Gradient $\\partial f / \\partial\\theta$', fontsize=12)
axes[1].set_xlabel('$\\theta$ (rad)'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('psr_verification.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n Observation: PSR matches analytic exactly; finite-difference has small but nonzero error.")

---
## Exercise 2: VQE on the 2-Qubit Transverse Ising Model

**Hamiltonian:** $H = Z_0 Z_1 + 0.5\, Z_0$

**Exact spectrum** (diagonal in the $Z$-basis):

| State | Energy |
|---|---|
| $|00\rangle$ | $+1.5$ |
| $|01\rangle$ | $-0.5$ |
| $|10\rangle$ | $-1.5$ ← ground state |
| $|11\rangle$ | $+0.5$ |

**Ansatz:** $\mathrm{CX}_{01}\,(R_y(\theta_0)\otimes R_y(\theta_1))\,|00\rangle$

**Student Experiments:**
1. Run VQE with `OPTIMIZER = 'COBYLA'`. Does it converge to $E_0 = -1.5$?
2. Change `OPTIMIZER = 'SPSA'`. Compare convergence speed (number of iterations).
3. Try `USE_PSR = True` to use the parameter-shift rule for gradients instead of COBYLA's built-in approximation. Compare the quality of the gradient.

In [ ]:
"""
VQE Module.
Implements the Variational Quantum Eigensolver for the 2-qubit Ising
Hamiltonian H = Z0Z1 + 0.5*Z0 using a hardware-efficient ansatz.
"""

# ── STUDENT EXPERIMENT ZONE ────────────────────────────────────────────────
OPTIMIZER    = 'COBYLA'   # 'COBYLA' or 'BFGS' or 'Nelder-Mead'
N_SHOTS      = 2000
N_RUNS       = 5          # Number of random restarts to show landscape
# ──────────────────────────────────────────────────────────────────────────


def build_ising_ansatz() -> QuantumCircuit:
    """
    Hardware-efficient ansatz for the 2-qubit Ising model.

    Circuit: CX_{01} * (Ry(θ₀) ⊗ Ry(θ₁)) |00⟩

    Analytical cost function:
        C(θ₀,θ₁) = ⟨Z₀Z₁⟩ + 0.5⟨Z₀⟩ = cos(θ₁) + 0.5·cos(θ₀)
    Minimum at θ₀=π, θ₁=π  →  C = -1 - 0.5 = -1.5
    """
    theta = ParameterVector('θ', 2)
    qc = QuantumCircuit(2, name='Ansatz')
    qc.ry(theta[0], 0)
    qc.ry(theta[1], 1)
    qc.cx(0, 1)
    return qc


def build_ising_hamiltonian() -> SparsePauliOp:
    """
    H = Z₀Z₁ + 0.5·Z₀
    Qiskit convention: string is qubit n-1 ... qubit 0
    'ZZ' means Z on q1 ⊗ Z on q0 (little-endian)
    """
    return SparsePauliOp.from_list([('ZZ', 1.0), ('ZI', 0.5)])


def psr_gradient_vqe(params: np.ndarray, estimator, circuit, hamiltonian) -> np.ndarray:
    """
    Compute gradient via parameter-shift rule for all parameters.

    For each parameter k:
        grad[k] = [C(θ + π/2·eₖ) - C(θ - π/2·eₖ)] / 2
    """
    grad = np.zeros_like(params)
    for k in range(len(params)):
        shift      = np.zeros_like(params)
        shift[k]   = math.pi / 2
        f_plus  = estimator.run([(circuit, hamiltonian, params + shift)]).result()[0].data.evs
        f_minus = estimator.run([(circuit, hamiltonian, params - shift)]).result()[0].data.evs
        grad[k] = (f_plus - f_minus) / 2
    return grad


# ── Set up circuit and Hamiltonian ─────────────────────────────────────────
ansatz = build_ising_ansatz()
H_ising = build_ising_hamiltonian()
estimator = Estimator()
display(ansatz.draw('mpl', style='iqp'))
print(f"Hamiltonian: {H_ising}")
print(f"Exact ground state energy: E₀ = -1.5  (|10⟩)")

# ── VQE optimisation ───────────────────────────────────────────────────────
cost_history  = []
params_history = []

def cost_function(params):
    job    = estimator.run([(ansatz, H_ising, params)])
    energy = float(job.result()[0].data.evs)
    cost_history.append(energy)
    params_history.append(params.copy())
    return energy

# Multiple random restarts
best_result = None
all_histories = []

for run in range(N_RUNS):
    cost_history   = []
    params_history = []
    x0 = np.random.uniform(0, 2*math.pi, 2)
    result = minimize(cost_function, x0, method=OPTIMIZER,
                      options={'maxiter': 200, 'rhobeg': 0.5})
    all_histories.append(cost_history.copy())
    if best_result is None or result.fun < best_result.fun:
        best_result = result
    print(f"  Run {run+1}: start={x0.round(2)}, "
          f"final E={result.fun:.4f}, "
          f"iters={result.nit}")

th0_opt, th1_opt = best_result.x
print(f"\nBest result: θ₀={th0_opt:.4f}, θ₁={th1_opt:.4f}, E={best_result.fun:.6f}")
print(f"Target E₀ = -1.5000,  Error = {abs(best_result.fun+1.5):.2e}")

# ── PSR gradient at the optimal point ─────────────────────────────────────
opt_params = best_result.x
grad = psr_gradient_vqe(opt_params, estimator, ansatz, H_ising)
print(f"\nPSR gradient at optimum: {grad.round(6)}")
print(f"(Expected: ≈ [0, 0] at the minimum)")

# ── Plot convergence ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, hist in enumerate(all_histories):
    axes[0].plot(hist, alpha=0.7, label=f'Run {i+1}')
axes[0].axhline(-1.5, color='red', linestyle='--', lw=2, label='$E_0 = -1.5$')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('$C(\\boldsymbol{\\theta})$')
axes[0].set_title(f'VQE Convergence ({OPTIMIZER})', fontsize=12)
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Cost landscape heatmap
tt0 = np.linspace(0, 2*math.pi, 80)
tt1 = np.linspace(0, 2*math.pi, 80)
TH0, TH1 = np.meshgrid(tt0, tt1)
COST = np.cos(TH1) + 0.5*np.cos(TH0)   # analytic formula
im = axes[1].contourf(TH0, TH1, COST, levels=30, cmap='coolwarm')
plt.colorbar(im, ax=axes[1], label='$C(\\theta_0, \\theta_1)$')
axes[1].set_xlabel('$\\theta_0$'); axes[1].set_ylabel('$\\theta_1$')
axes[1].set_title('Cost Landscape with Optimal Point', fontsize=12)
axes[1].scatter([th0_opt % (2*math.pi)], [th1_opt % (2*math.pi)],
                c='lime', s=120, zorder=5, marker='*', label='VQE optimum')
axes[1].scatter([math.pi], [math.pi],
                c='black', s=80, zorder=5, marker='x', label='True minimum')
axes[1].legend()

plt.tight_layout()
plt.savefig('vqe_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Exercise 3: QAOA for MaxCut on $C_4$

**Graph:** 4-node cycle $C_4$, edges $\{(0,1),(1,2),(2,3),(3,0)\}$.

**Optimal solution:** Partition $\{0,2\}$ vs $\{1,3\}$, cutting all 4 edges.  
This corresponds to bitstrings $|0101\rangle$ and $|1010\rangle$.

**Cost Hamiltonian:**
$H_C = 2I - \frac{1}{2}(Z_0Z_1 + Z_1Z_2 + Z_2Z_3 + Z_3Z_0)$

**QAOA circuit (one layer):**
For each edge $(u,v)$: $\mathrm{CNOT}_{uv}\,R_z(2\gamma)\,\mathrm{CNOT}_{uv}$ (implements $e^{-i\gamma Z_u Z_v}$)
Then: $R_x(2\beta)$ on all qubits (mixer)

**Student Experiments:**
1. Run the cell. Observe how $P(|0101\rangle) + P(|1010\rangle)$ increases with $p$.
2. Change `P_MAX = 5`. Does the approximation ratio keep improving?
3. The variable `USE_WARM_START` initialises each layer $p$ from the optimal parameters of $p-1$. Try `True` and `False` and compare convergence speed.

In [ ]:
"""
QAOA MaxCut Module.
Implements QAOA for MaxCut on the 4-cycle C4 with p ∈ {1,2,3} layers.
Compares approximation ratio, probability concentration, and convergence.
"""

# ── STUDENT EXPERIMENT ZONE ────────────────────────────────────────────────
P_MAX        = 3      # Maximum number of QAOA layers (try 5 for extended study)
USE_WARM_START = True # Initialise layer k from optimal params of layer k-1
N_SHOTS_QAOA   = 4000
# ──────────────────────────────────────────────────────────────────────────


EDGES_C4   = [(0, 1), (1, 2), (2, 3), (3, 0)]
MAXCUT_OPT = 4   # known optimal MaxCut value for C4


def build_qaoa_circuit(n: int, edges: list, p: int) -> tuple:
    """
    Constructs a QAOA circuit for MaxCut.

    Args:
        n:     Number of qubits (nodes).
        edges: List of (u, v) pairs.
        p:     Number of QAOA layers.

    Returns:
        (circuit, all_parameters) — the QuantumCircuit and its
        ParameterVector in order [γ₀, γ₁, ..., β₀, β₁, ...].
    """
    g = ParameterVector('\u03b3', p)
    b = ParameterVector('\u03b2', p)
    qc = QuantumCircuit(n, name=f'QAOA p={p}')

    # Initial state |+>^n
    qc.h(range(n))
    qc.barrier()

    for k in range(p):
        # ── Cost unitary: e^{-iγ H_C} ──────────────────────────────────────
        # Each ZZ term: CNOT – Rz(2γ) – CNOT implements e^{-iγ ZiZj}
        for (u, v) in edges:
            qc.cx(u, v)
            qc.rz(2 * g[k], v)
            qc.cx(u, v)
        qc.barrier()
        # ── Mixer: e^{-iβ H_B} = ⊗_i Rx(2β) ──────────────────────────────
        qc.rx(2 * b[k], range(n))
        qc.barrier()

    return qc, list(g) + list(b)


def build_maxcut_hamiltonian(n: int, edges: list) -> SparsePauliOp:
    """
    H_C = Σ_{(i,j)} (I - Z_i Z_j) / 2
    We maximise H_C ≡ minimise -H_C.
    """
    terms = []
    for (u, v) in edges:
        zz_str = list('I' * n)
        zz_str[u] = 'Z'
        zz_str[v] = 'Z'
        zz_str.reverse()   # Qiskit little-endian
        terms.append((''.join(zz_str), -0.5))
        terms.append(('I' * n,          0.5))
    return SparsePauliOp.from_list(terms).simplify()


H_maxcut  = build_maxcut_hamiltonian(4, EDGES_C4)
estimator = Estimator()
sampler   = Sampler()
sim       = AerSimulator()

results_by_p  = {}
opt_params    = {}

for p in range(1, P_MAX + 1):
    qaoa_circ, params_list = build_qaoa_circuit(4, EDGES_C4, p)
    n_params = len(params_list)

    # Warm-start: extend previous optimal params
    if USE_WARM_START and p > 1:
        prev = opt_params[p-1]
        x0 = np.concatenate([
            np.append(prev[:p-1], [0.1]),    # gammas
            np.append(prev[p-1:], [0.1])     # betas
        ])
    else:
        x0 = np.random.uniform(0, math.pi/2, n_params)

    iters = []

    def neg_cut(params):
        job = estimator.run([(qaoa_circ, H_maxcut, params)])
        val = float(job.result()[0].data.evs)
        iters.append(val)
        return -val   # maximise cut ≡ minimise negative

    res = minimize(neg_cut, x0, method='COBYLA',
                   options={'maxiter': 300 * p, 'rhobeg': 0.5})

    opt_params[p] = res.x
    approx_ratio  = -res.fun / MAXCUT_OPT

    # Sample the optimal circuit to get probability distribution
    opt_circ = qaoa_circ.copy()
    opt_circ = opt_circ.assign_parameters(
        dict(zip(params_list, res.x)))
    opt_circ.measure_all()
    counts = sim.run(opt_circ, shots=N_SHOTS_QAOA).result().get_counts()

    # Parse to 16-element prob vector
    probs = np.zeros(16)
    for state, cnt in counts.items():
        idx = int(state.replace(' ', ''), 2)
        probs[idx] = cnt / N_SHOTS_QAOA

    results_by_p[p] = {
        'probs':        probs,
        'approx_ratio': approx_ratio,
        'max_cut':      -res.fun,
        'opt_params':   res.x,
        'history':      iters
    }

    # Probability on optimal bitstrings |0101⟩=5 and |1010⟩=10
    p_opt = probs[5] + probs[10]
    print(f"p={p}: MaxCut≈{-res.fun:.3f}/{MAXCUT_OPT}, "
          f"ratio={approx_ratio:.3f}, "
          f"P(|0101⟩+|1010⟩)={p_opt*100:.1f}%, "
          f"iters={res.nit}")


# ── Summary plot ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 4 * P_MAX))
gs  = gridspec.GridSpec(P_MAX, 2, hspace=0.4, wspace=0.3)

states = [format(i, '04b') for i in range(16)]
colors = ['#2e7d32' if i in (5, 10) else '#1565c0' for i in range(16)]

for row, p in enumerate(range(1, P_MAX+1)):
    r = results_by_p[p]

    # Bar chart of probabilities
    ax1 = fig.add_subplot(gs[row, 0])
    bars = ax1.bar(range(16), r['probs'], color=colors, edgecolor='white', lw=0.5)
    ax1.axhline(1/16, color='gray', linestyle='--', lw=1, label='Uniform (1/16)')
    ax1.set_xticks(range(16))
    ax1.set_xticklabels(states, rotation=90, fontsize=7)
    ax1.set_title(f'p={p}: Probability Distribution\n'
                  f'Approx. ratio = {r["approx_ratio"]:.3f}  '
                  f'|0101⟩+|1010⟩ = {(r["probs"][5]+r["probs"][10])*100:.1f}%',
                  fontsize=10)
    ax1.set_ylabel('Probability')
    ax1.legend(fontsize=8)

    # Convergence history
    ax2 = fig.add_subplot(gs[row, 1])
    ax2.plot(r['history'], 'b-', lw=1.5, alpha=0.8)
    ax2.axhline(MAXCUT_OPT, color='red', linestyle='--', lw=2, label=f'MaxCut = {MAXCUT_OPT}')
    ax2.set_title(f'p={p}: Optimiser Convergence (⟨H_C⟩)', fontsize=10)
    ax2.set_xlabel('Iteration'); ax2.set_ylabel('$\\langle H_C\\rangle$')
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.suptitle(f'QAOA MaxCut on $C_4$ — p ∈ {{1,...,{P_MAX}}}', fontsize=13, y=1.01)
plt.savefig('qaoa_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n── Approximation ratio summary ──")
for p in range(1, P_MAX+1):
    r = results_by_p[p]
    bar = '█' * int(r['approx_ratio'] * 20)
    print(f"  p={p}: {bar} {r['approx_ratio']*100:.1f}%")

---
## Bridge to Lab 5: Preview of the Portfolio QUBO

The portfolio optimization problem from Lab 5 has the form:

$$\min_{x \in \{0,1\}^n} x^T Q\, x, \qquad Q_{ij} = \lambda\,\Sigma_{ij} - \mu_i\,\delta_{ij}$$

This maps to a QAOA Hamiltonian via $x_i = (1-Z_i)/2$. The cell below shows the structural
equivalence: the same `build_qaoa_circuit` function works with any Hamiltonian.

In [ ]:
"""
Bridge to Lab 5: QUBO → Hamiltonian mapping.
Shows how a 4-asset portfolio QUBO maps to a QAOA Hamiltonian
using the same circuit structure built in Exercise 3.
"""

def qubo_to_ising(Q: np.ndarray, lam_penalty: float = 1.0,
                  K: int = 2) -> SparsePauliOp:
    """
    Convert a QUBO matrix Q (n×n) to an Ising Hamiltonian.

    Substitution: x_i = (1 - Z_i) / 2
    Adds penalty λ(Σ x_i - K)² to enforce budget constraint.

    Args:
        Q:           QUBO cost matrix (risk-return trade-off).
        lam_penalty: Penalty weight for budget constraint.
        K:           Target number of assets to select.

    Returns:
        SparsePauliOp representing H_portfolio.
    """
    n     = Q.shape[0]
    terms = []

    # Quadratic terms from x_i = (1-Z_i)/2
    for i in range(n):
        for j in range(n):
            if abs(Q[i,j]) < 1e-10:
                continue
            coeff = Q[i,j] / 4
            # x_i x_j = (1-Z_i)(1-Z_j)/4 = (I - Z_i - Z_j + Z_iZ_j)/4
            terms.append(('I'*n,           coeff))
            zi = list('I'*n); zi[i]='Z'; zi.reverse()
            zj = list('I'*n); zj[j]='Z'; zj.reverse()
            zij = list('I'*n); zij[i]='Z'; zij[j]='Z'; zij.reverse()
            if i != j:
                terms.append((''.join(zi), -coeff))
                terms.append((''.join(zj), -coeff))
                terms.append((''.join(zij),  coeff))
            else:
                terms.append((''.join(zi), -2*coeff))

    # Budget penalty: λ(Σ (1-Z_i)/2 - K)² = λ(n/2 - K - Σ Z_i/2)²
    shift = n/2 - K
    for i in range(n):
        zi = list('I'*n); zi[i]='Z'; zi.reverse()
        terms.append((''.join(zi),  lam_penalty * shift / 1))
        terms.append((''.join(zi), -lam_penalty / 4))
        for j in range(i+1, n):
            zij = list('I'*n); zij[i]='Z'; zij[j]='Z'; zij.reverse()
            terms.append((''.join(zij), lam_penalty / 4))

    return SparsePauliOp.from_list(terms).simplify()


# ── Example: 4-asset portfolio ─────────────────────────────────────────────
# Covariance matrix (risk) — symmetric, positive definite
Sigma = np.array([
    [0.10, 0.03, 0.01, 0.02],
    [0.03, 0.08, 0.04, 0.01],
    [0.01, 0.04, 0.12, 0.02],
    [0.02, 0.01, 0.02, 0.07]
])
mu  = np.array([0.05, 0.08, 0.04, 0.06])   # expected returns
lam = 2.0                                   # risk-aversion

# QUBO: Q = λΣ - diag(μ)
Q_portfolio = lam * Sigma - np.diag(mu)

H_portfolio = qubo_to_ising(Q_portfolio, lam_penalty=3.0, K=2)

print("4-Asset Portfolio QUBO matrix Q = λΣ - diag(μ):")
print(np.round(Q_portfolio, 4))
print(f"\nPortfolio Hamiltonian H_portfolio ({len(H_portfolio)} Pauli terms):")
print(H_portfolio)

print("\n In Lab 5, you will run QAOA with this Hamiltonian as H_C.")
print("   The circuit structure is identical to Exercise 3 —")
print("   only H_C changes. The classical optimizer and PSR gradients are the same.")

# Show the circuit structure is reusable
qaoa_portfolio, _ = build_qaoa_circuit(4, EDGES_C4, p=1)
print(f"\nSame QAOA circuit (p=1), but with H_portfolio as the Hamiltonian:")
display(qaoa_portfolio.draw('mpl', style='iqp', fold=60))

---
## OPTIONAL: Noise Analysis — VQE on Real Hardware

The cell below runs the VQE Ising circuit on a **noisy simulator** using the
device characteristics of a real IBM QPU. This shows how gate errors and
decoherence degrade the cost function estimate.

Fill in your IBM Quantum API token to optionally run on a real QPU.

In [ ]:
"""
Noise Analysis Module.
Compares VQE convergence under noiseless vs noisy simulation,
and optionally runs on real IBM Quantum hardware.
"""
from qiskit_aer.noise import NoiseModel
from qiskit_aer.noise.errors import depolarizing_error, thermal_relaxation_error

def build_simple_noise_model(p1q: float = 0.001, p2q: float = 0.01) -> NoiseModel:
    """
    Constructs a simple depolarizing noise model.

    Args:
        p1q: Single-qubit gate error probability.
        p2q: Two-qubit gate error probability.
    """
    noise_model = NoiseModel()
    error_1q    = depolarizing_error(p1q, 1)
    error_2q    = depolarizing_error(p2q, 2)
    noise_model.add_all_qubit_quantum_error(error_1q, ['ry', 'rz', 'rx', 'h'])
    noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])
    return noise_model


ansatz_noise = build_ising_ansatz()
noise_model  = build_simple_noise_model(p1q=0.001, p2q=0.01)
noisy_sim    = AerSimulator(noise_model=noise_model)
noisy_est    = Estimator(backend=noisy_sim)

# VQE under noise
hist_noisy = []
x0 = np.array([math.pi * 0.9, math.pi * 1.1])

def cost_noisy(params):
    job = noisy_est.run([(ansatz_noise, H_ising, params)])
    val = float(job.result()[0].data.evs)
    hist_noisy.append(val)
    return val

res_noisy = minimize(cost_noisy, x0, method='COBYLA',
                     options={'maxiter': 150})

# VQE noiseless for comparison (same starting point)
hist_clean = []
def cost_clean(params):
    job = estimator.run([(ansatz_noise, H_ising, params)])
    val = float(job.result()[0].data.evs)
    hist_clean.append(val)
    return val

res_clean = minimize(cost_clean, x0.copy(), method='COBYLA',
                     options={'maxiter': 150})

print("\n=== Noise Impact on VQE ===")
print(f"True ground state:       E₀ = -1.5000")
print(f"Noiseless convergence:   E  = {res_clean.fun:.4f}  (error {abs(res_clean.fun+1.5):.2e})")
print(f"Noisy convergence:       E  = {res_noisy.fun:.4f}  (error {abs(res_noisy.fun+1.5):.2e})")
print(f"Noise overhead:          ΔE = {abs(res_noisy.fun - res_clean.fun):.4f}")

plt.figure(figsize=(8, 4))
plt.plot(hist_clean, 'b-', lw=2, label='Noiseless', alpha=0.9)
plt.plot(hist_noisy, 'r-', lw=2, label=f'Noisy (p_2q={0.01})', alpha=0.9)
plt.axhline(-1.5, color='black', linestyle='--', lw=1.5, label='$E_0 = -1.5$')
plt.xlabel('Iteration'); plt.ylabel('$C(\\boldsymbol{\\theta})$')
plt.title('VQE Under Noise: Noiseless vs Depolarising Model', fontsize=12)
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('vqe_noise.png', dpi=150, bbox_inches='tight')
plt.show()